In [1]:
df = spark.read.csv('delivery_delay_dataset.csv', header=True, inferSchema=True)
df.show(5)

+-----------+----+-------------+------+
|order_count|rain|delivery_time| delay|
+-----------+----+-------------+------+
|         66|   0|         58.4|  Late|
|         50|   1|         45.0|  Late|
|         37|   0|         36.8|OnTime|
|         79|   1|         72.6|  Late|
|         26|   0|         27.4|OnTime|
+-----------+----+-------------+------+
only showing top 5 rows



In [2]:
df.printSchema()

root
 |-- order_count: integer (nullable = true)
 |-- rain: integer (nullable = true)
 |-- delivery_time: double (nullable = true)
 |-- delay: string (nullable = true)



In [3]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


label_indexer = StringIndexer(
    inputCol="delay",
    outputCol="label"
)

feature_cols = [
    "order_count",
    "rain",
    "delivery_time"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20
)

pipeline = Pipeline(stages=[
    label_indexer,
    assembler,
    rf
])

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train_df)

predictions = model.transform(test_df)

predictions.select(
    "order_count",
    "rain",
    "delivery_time",
    "delay",
    "prediction"
).show(10)

+-----------+----+-------------+------+----------+
|order_count|rain|delivery_time| delay|prediction|
+-----------+----+-------------+------+----------+
|          5|   0|         13.0|OnTime|       1.0|
|          5|   0|         21.0|OnTime|       1.0|
|          5|   0|         26.0|OnTime|       1.0|
|          5|   0|         32.0|OnTime|       1.0|
|          5|   1|         38.0|OnTime|       1.0|
|          5|   1|         42.0|  Late|       0.0|
|          6|   0|         19.4|OnTime|       1.0|
|          6|   0|         30.4|OnTime|       1.0|
|          6|   1|         32.4|OnTime|       1.0|
|          6|   1|         37.4|OnTime|       1.0|
+-----------+----+-------------+------+----------+
only showing top 10 rows



In [4]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print(f"\nModel Accuracy: {accuracy:.4f}")


Model Accuracy: 0.9964


In [6]:
#model.save("delivery_delay_model")

## Streaming

In [12]:
from pyspark.sql.functions import split, col, avg, count
from pyspark.ml import PipelineModel

model = PipelineModel.load("delivery_delay_model")

raw_stream = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

split_cols = split(raw_stream.value, ",")

stream_df = raw_stream.select(
    split_cols.getItem(0).cast("integer").alias("order_count"),
    split_cols.getItem(1).cast("integer").alias("rain"),
    split_cols.getItem(2).cast("double").alias("delivery_time")
)

predictions = model.transform(stream_df)

output = predictions.select(
    "order_count",
    "rain",
    "delivery_time",
    "prediction"
)

aggregations = predictions.groupBy("prediction").agg(
    count("*").alias("total_orders"),
    avg("delivery_time").alias("avg_delivery_time")
)


prediction_query = output.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

aggregation_query = aggregations.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

spark.streams.awaitAnyTermination()

26/05/15 01:50:59 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/05/15 01:50:59 WARN StringIndexerModel: Input column delay does not exist during transformation. Skip StringIndexerModel for this column.
26/05/15 01:50:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d78de644-230a-4701-97e4-5e7551575f4d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/15 01:50:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/15 01:50:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7826d34d-e68b-4b4c-943f-93cf9e0c6

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+----+-------------+----------+
|order_count|rain|delivery_time|prediction|
+-----------+----+-------------+----------+
+-----------+----+-------------+----------+



-------------------------------------------
Batch: 0
-------------------------------------------
+----------+------------+-----------------+
|prediction|total_orders|avg_delivery_time|
+----------+------------+-----------------+
+----------+------------+-----------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+----+-------------+----------+
|order_count|rain|delivery_time|prediction|
+-----------+----+-------------+----------+
|         40|   1|         52.0|       0.0|
|         15|   0|         20.0|       1.0|
+-----------+----+-------------+----------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+----+-------------+----------+
|order_count|rain|delivery_time|prediction|
+-----------+----+-------------+----------+
|         15|   0|         20.0|       1.0|
|         80|   1|         60.0|       0.0|
|         20|   1|         36.0|       1.0|
+--

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/asish-jose/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/asish-jose/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 